In [ ]:

import pandas as pd
from pymdp import utils
import numpy as np
from scipy.stats import norm
import math
from pymdp.maths import softmax

import jax.numpy as jnp
import jax.tree_util as jtu
from jax import random as jr, jit
from pymdp.jax.agent import Agent as AIFAgent
from functools import partial
from equinox import tree_at

import jax

import csv


In [ ]:
from jax import nn
def calc_action_selection_prob(agent,neg_efe):
    neg_efe_tmp=neg_efe.flatten()
    q_pi=nn.softmax(agent.gamma * neg_efe_tmp)
    q_pi_record=q_pi[:-1]
    q_pi=jnp.expand_dims(q_pi, -2)
    q_pi_record=jnp.expand_dims(q_pi_record, -2)
    return q_pi,q_pi_record

In [ ]:
def f32(x): 
    return jnp.asarray(x, dtype=jnp.float32).reshape((1,))

In [ ]:
##dummy parameter
sa1=0.001
sa2=0.001
sa3=0
sa4=12

sa=[sa1, sa2,sa3,sa4]


sb1=0.001

sb2=sb1
sb3=sb1
sb4=sb3
sb=[sb1, sb2,sb3,sb4]
sA1=sa1
sA2=sA1
sA3=sA1
sA4=sA1
sA=[sA1, sA2,sA3,sA4]

lr_pA=f32(1)
fr_pA=f32(1)
lr_pB=f32(5*sc)
fr_pB=f32(0.5)
num_slot=2
num_iter=32
tau=f32(0.1)

In [ ]:

@partial(jit, static_argnames=['batch_size','num_history'])
def update_agent_action_sampling(agents, outcomes, actions, infer_args, batch_keys, beliefs,actions_t, reward_fuction=None,batch_size=1, num_history=1000):

    q_pi, neg_efe, pbs,pbs_st, pkld, pfe, oRisk, pbs_pA, pbs_pB,_,I_B_o,Hqs = agents.infer_policies_efe_curiosity(beliefs)
    Hqo=pfe-pkld
    
    Pv=-oRisk-Hqo

    batch_keys = jr.split(batch_keys[0], batch_size)

    
    q_pi,q_pi_record=calc_action_selection_prob(agents,pbs+Pv)


    if actions is not None:
        actions = jnp.concatenate([actions, jnp.expand_dims(actions_t, -2)], -2)##次の推論でsampleactionしたものが使われる？
    else:
        actions = jnp.expand_dims(actions_t, -2)
    outcomes = jtu.tree_map( lambda x: x[:, -num_history:], outcomes)
    beliefs = jtu.tree_map( lambda x: x[:, -num_history:], beliefs)
    actions = jtu.tree_map( lambda x: x[:,-num_history:], actions)

    infer_args = agents.update_empirical_prior(actions_t, beliefs)
    return agents, outcomes, actions, infer_args, batch_keys, neg_efe, pbs, pbs_pA, pbs_pB,  Hqo, pkld, pfe, oRisk,q_pi_record,I_B_o,Hqs,pbs_st

In [ ]:
@partial(jit, static_argnames=['batch_size','num_history'])
def update_agent_inference(agents, outcomes, actions, infer_args, batch_keys,expected_states, batch_size=1, num_history=1000):

    beliefs,err, vfe,kld2, bs, un, qs_1step, err_1step, vfe_1step, kld2_1step, bs_1step, un_1step = agents.infer_states_vfe_set_prior(outcomes, infer_args[0], past_actions=actions, qs_hist=infer_args[1],expected_states=expected_states, tau=tau)
    kld = agents.calc_KLD_past_currentqs(expected_states, infer_args[1], beliefs)
    outcomes = jtu.tree_map( lambda x: x[:, -num_history:], outcomes)
    beliefs_temp = jtu.tree_map( lambda x: x[:, -num_history:], beliefs)
    actionstmp = jtu.tree_map( lambda x: x[:,-num_history:], actions)

    beliefs_last = jtu.tree_map( lambda x: x[:, -1:], beliefs_temp) # take the last belief
    outcomes_last = jtu.tree_map( lambda x: x[:, -1:], outcomes) # take the last outcome
    applied_actions_last = jtu.tree_map( lambda x: x[:,-1:], actionstmp) # take the last applied action#applied_actions_last = jtu.tree_map( lambda x: x[:,-2:-1], actions) 
    beliefs_last_pair = jtu.tree_map( lambda x: x[:, -2:], beliefs_temp) # take the last two beliefs

    if applied_actions_last is None:
        q_pi, neg_efe, pbs, pkld, pfe, oRisk, pbs_pA, pbs_pB,I_B_o,I_B_o_se = agents.infer_policies_efe(beliefs)##
        batch_keys = jr.split(batch_keys[0], batch_size)
        actions_t = agents.sample_action(q_pi, rng_key=batch_keys)
        if actions is not None:
            actionstmp = jnp.concatenate([actionstmp, jnp.expand_dims(actions_t, -2)], -2)
        else:
            actionstmp = jnp.expand_dims(actions_t, -2)
        applied_actions_last = jtu.tree_map( lambda x: x[:,-2:-1], actionstmp)
    agents = tree_at(lambda x: x.D, agents, jtu.tree_map(lambda x: x[:, 0], beliefs_temp)) 
    vfe =jnp.array(vfe)
    vfe_1step =jnp.array(vfe_1step)
    
    bs =jnp.array(bs)
    bs_1step =jnp.array(bs_1step)
    
    un =jnp.array(un)
    un_1step =jnp.array(un_1step)
    return agents, outcomes, actions, infer_args, batch_keys,  vfe,  bs, un, kld, beliefs, vfe_1step


In [ ]:
def selected_policy_entropy(entropy, selected_slot):

    selected_slot=jnp.array(selected_slot)

    entropy=jnp.array(entropy)
    entropy=entropy[:,:-1]

    selected_elements = []
    for t in range(len(selected_slot)):
        slot_index = selected_slot[t] - 1  
        if 0 <= slot_index < 4:
            selected_elements.append(entropy[slot_index][t])
    return selected_elements

In [16]:
def clip(s,eps):
    if s==0:
        s=s+eps
    return s

In [ ]:
p_id=["p37","p38","p39","p40"]
run_num=1
loopsize=len(p_id)
for p in range(loopsize):
  output_file = f"block_results_{p_id[p]}_{run_num}.csv"

  records = []

  with open(output_file, mode="w", newline="") as f:
      writer = csv.writer(f)
      writer.writerow(["p_id","run_num","blocknum","trial","condition","sa1","sa2","sb","choice","reward","PBS_diff", "PBS_st_diff",
        "PBS_total_diff",
        "H_qo_diff",
        "Pragmaticvalue_diff",
        "pBS_pA_diff",
        "pBS_pB_diff",
        "Q_pi_s",
        "PBS_select",
        "PBS_st_select",
        "PBS_total_select",
        "Hqo_select",
        "I_B_o_select",
        "Pragmaticvalue_select",
        "PBS_0",
        "PBS_1",
        "PBS_st_0",
        "PBS_st_1",
        "H_qo_0",
        "H_qo_1",
        "Pragmaticvalue_0",
        "Pragmaticvalue_1",
        "pBS_pA_0",
        "pBS_pA_1",
        "pBS_pB_0",
        "pBS_pB_1",
        "prior_VFE",
        "posterior_VFE",
        "BS",
        "KLD",
        "H_qs_diff",
        "H_qs_0",
        "H_qs_1",
        "Hqs_select"])


  file_path = f"/log/bandit_{p_id[p]}_0{run_num}.csv"

  df = pd.read_csv(file_path)


  sa1_list=df["arm_left_sa"].tolist()
  sa2_list=df["arm_right_sa"].tolist()
  sb1_list=df["arm_left_sb"].tolist()
  print(f"sa1=",sa1_list)
  print(f"sa2=",sa2_list)
  slots = df["choice"].tolist()
  slots = [(3 if x == 0 else x) - 1 for x in slots]
  rewards = df["reward_int"].tolist()#1足す？
  print("slots:", slots)
  print("rewards:", rewards)
  Timerange=4
  u_data=slots
  o_data=rewards

  block=df["block"].tolist()
  trial_in_block=df["trial_in_block"].tolist()
  condition_list=df["condition"].tolist()

  num_block=int(len(slots)/4)
  print(num_block)

  seedjax=43
  
  
  
  use_param_info_gain=False
  learn_A=False
  learn_B=False
  e=0.001
  Timerange=4
 
  
  num_dim=100
  cgrad=15
  b=0.06
  x_mid=65
  alpha=1
  
  num_obs=[]
  num_states=[]
  num_controls=[]
  for i in range(num_slot):
      num_obs.append(num_dim+1)
      num_states.append(num_dim)
      num_controls.append(num_slot+1)
  num_obs.append(num_slot+1)
  num_states.append(num_slot+1)
  num_controls.append(num_slot+1)
  num_states_A = [num_dim,  num_slot+1]
  Ns1=num_states[0]
  policies = np.array([[[i] * (num_slot +1)] for i in range(num_slot+1)])
  num_factors = len(num_states)
  D = utils.obj_array(num_factors)
  for i in range(num_slot):
      D[i] = np.ones(num_dim)/num_dim

  D[num_slot] = np.array([1/num_slot for _ in range(num_slot)]+[0])


  l = np.arange(0,num_dim,1)                 
  A_shapes = [[o_dim] + num_states_A for o_dim in num_obs[0:num_slot]]
  A_shapes.append([num_slot+1,num_slot+1])
  # initialize the A array to all 0's
  A_array = utils.obj_array_zeros(A_shapes)
  for j in range(num_slot):
      for m in range(0,num_states[j],1):
          for n in range(num_slot+1):
              if n==j:
                  #print(n)
                  A_array[j][0,m,n]=0 
                  A_array[j][1:(num_dim+1),m,n]=  norm.pdf(l,m,sa[j])+e 
              else:
                  A_array[j][0,m,n]=1
          

  for q in range(0,num_states[num_slot],1):
      
      A_array[num_slot][q,q]=1

  A_array = utils.norm_dist_obj_arr(A_array)         

  B_shapes = [[s_dim, s_dim, num_controls[f]] for f, s_dim in enumerate(num_states)]
  B_array = utils.obj_array_zeros(B_shapes)

  for k in range(0,num_slot,1):
      for u in range(0,num_controls[k],1):
          for st in range(0,num_states[k],1): 
              B_array[k][:,st,u]=norm.pdf(l,st,sb[k])+e


  for controls in range(0,num_states[num_slot],1):
      B_array[num_slot][controls,:,controls]=[1] * (num_slot+1)
  B_array = utils.norm_dist_obj_arr(B_array)


  C_shapes = [[o_dim] for o_dim in num_obs]
  C_vector = utils.obj_array_zeros(C_shapes)

  eps = 1e-12
  for k in range(0, num_slot, 1):
      for m in range(0, num_obs[k], 1):
          x = m
          t = b * (x - x_mid)
          sig = 1.0 / (1.0 + math.exp(-t))  
          C_vector[k][m] = cgrad * sig 

  for i in range(num_slot):
      C_vector[i]=np.log(softmax(C_vector[i]))

  batch_size = 1 # number of agents
  #jax
  A_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(A_array))
  B_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(B_array))
  C_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(C_vector))
  D_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(D))

  batch_keys = jr.split(jr.PRNGKey(0), batch_size)
  A_dependencies = [[i, num_slot] for i in range(num_slot)] + [[num_slot]]
  B_dependencies=None

  agents = AIFAgent(A=A_jax, B=B_jax, C=C_jax, D=D_jax, E=None, pA=None, pB=None,  inference_algo="mmp", learn_A=learn_A, learn_B=learn_B, learn_C=False, learn_D=False, learn_E=False, A_dependencies=A_dependencies, B_dependencies=B_dependencies, policies=policies,gamma=1., alpha=alpha, action_selection="stochastic", policy_len=1, use_utility=True, use_states_info_gain=True, use_param_info_gain=use_param_info_gain, use_inductive=False, onehot_obs=False, sampling_mode="full",num_iter=num_iter)
  for blocknum in range(num_block):
    data_idx=0+4*blocknum

    condition=condition_list[data_idx]
    sa1=sa1_list[data_idx]
    sa2=sa2_list[data_idx]
    sb1=sb1_list[data_idx]
    eps=0.001
    sa1=clip(sa1,eps)
    sa2=clip(sa2,eps)
    sb1=clip(sb1,eps)
    sa3=0
    sa4=12
    
    sa=[sa1, sa2,sa3,sa4]

    sb2=sb1
    sb3=sb1
    sb4=sb3
    sb=[sb1, sb2,sb3,sb4]
    A_array = utils.obj_array_zeros(A_shapes)
    for j in range(num_slot):
        for m in range(0,num_states[j],1):
            for n in range(num_slot+1):
                if n==j:

                    A_array[j][0,m,n]=0 

                    A_array[j][1:(num_dim+1),m,n]=  norm.pdf(l,m,sa[j])+e 
                else:
                    A_array[j][0,m,n]=1
            

    for q in range(0,num_states[num_slot],1):
        
        A_array[num_slot][q,q]=1

    A_array = utils.norm_dist_obj_arr(A_array)         

    B_shapes = [[s_dim, s_dim, num_controls[f]] for f, s_dim in enumerate(num_states)]
    B_array = utils.obj_array_zeros(B_shapes)

    for k in range(0,num_slot,1):
        for u in range(0,num_controls[k],1):
            for st in range(0,num_states[k],1): 
                B_array[k][:,st,u]=norm.pdf(l,st,sb[k])+e


    for controls in range(0,num_states[num_slot],1):
        B_array[num_slot][controls,:,controls]=[1] * (num_slot+1)
    B_array = utils.norm_dist_obj_arr(B_array)
    
    #jax
    A_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(A_array))
    B_jax = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(B_array))
    agents = tree_at(lambda x: x.A, agents, A_jax) 
    agents = tree_at(lambda x: x.B, agents, B_jax)  
    agents = tree_at(lambda x: x.D, agents, D_jax) 
    T=Timerange
    key = jr.PRNGKey(seedjax)
    batch_keys = jr.split(key, T)
    slot_counts = [0 for _ in range(num_slot+1)]
    array_selected_slot=[]
    EFE=[]
    prior_VFE=[]
    posterior_VFE=[]
    VFE=[]
    PBS=[]
    PKLD=[]
    PFE=[]
    Risk=[]
    Ambiguity=[]
    Pragmaticvalue=[]
    KLD=[]
    BS=[]
    pBS_pA=[]
    pBS_pB=[]
    H_qo=[]
    Q_pi=[]
    Q_pi_s=[]
    PBS_st=[]
    prediction_performance=[]
    H_qs=[]

    prior_beliefs=jtu.tree_map(lambda x: jnp.expand_dims(x, -2), agents.D)
    q_pi, neg_efe, pbs,pbs_st, pkld, pfe, oRisk, pbs_pA, pbs_pB,_,I_B_o,Hqs = agents.infer_policies_efe_curiosity(prior_beliefs)
    Hqo=pfe-pkld
    Hqs=jnp.array([Hqs[0,0,0],Hqs[0,0,1]])
    H_qs.append(Hqs)
    Pv=-oRisk-Hqo
    q_pi,q_pi_record=calc_action_selection_prob(agents,Pv+pbs)
    batch_keys = jr.split(batch_keys[0], batch_size)
    uidx=u_data[0+blocknum*4]
    oidx=o_data[0+blocknum*4]
    obs=[0 for _ in range(num_slot+1)]
    obs[-1]=uidx
    if oidx>100:
      oidx=100
    elif oidx<0:
      oidx=0
    obs[uidx]=oidx
    actions_t=jnp.array([uidx for _ in range(num_slot+1)])
    actions_t=jnp.expand_dims(actions_t, -2)
    print("selected action",actions_t)
    first_action=actions_t
    obs = jtu.tree_map(lambda x: jnp.expand_dims(x, -1).astype(jnp.int32), obs)
    if uidx==2:
      Q_pi_s.append(np.nan)
    else: 
      Q_pi_s.append(q_pi_record[0][uidx])
    Q_pi.append(q_pi_record)
    PBS_st.append(pbs_st)
    obs_record=[obs]
    last_action=first_action[0][num_slot]
    D[num_slot][first_action[0][num_slot]] = 1
    D_jax_tmp = jtu.tree_map(lambda x: jnp.broadcast_to(x, (batch_size,) + x.shape), list(D))
    agents = tree_at(lambda x: x.D, agents, D_jax_tmp) 

    PBS.append(pbs)
    PKLD.append(pkld)
    PFE.append(pfe)
    Risk.append(oRisk)  
    ambi=pfe-pbs-pkld
    Ambiguity.append(ambi)
    Pragmaticvalue.append(Pv)
    H_qo.append(Hqo)
    EFE.append(-1*neg_efe)
    pBS_pA.append(pbs_pA)
    pBS_pB.append(I_B_o)
    selected_slot = int(actions_t[0][num_slot]) 
    slot_counts[selected_slot] += 1
    array_selected_slot.append(first_action[0][num_slot]+1)
    beliefs=None
    for t in range(T):
      if t == 0:
          
          actions_t = None # no action available at the first time step
          actions = None # no action available at the first time step
          prob_pi = None
          infer_args = (agents.D, None,)
          outcome_t  = None
          outcomes = None
          expected_state =agents.D 
      else:
          print("selected action",actions_t)
          expected_state = agents.compute_expected_state(actions_t, infer_args[1])
          


      batch_keys = jr.split(batch_keys[0], batch_size)
      outcome_t  = jtu.tree_map(lambda x: jnp.expand_dims(x, -1), obs)

      num_history=4
      if outcomes is None:
          outcomes = outcome_t
      else:
          outcomes = jtu.tree_map(lambda prev_o, new_o: jnp.concatenate([prev_o, new_o], -1), outcomes, outcome_t) 
      

      agents, outcomes, actions, infer_args, batch_keys,  posterior_vfe,  bs, un, kld, beliefs,prior_vfe = update_agent_inference(
        agents, 
        outcomes, 
        actions, 
        infer_args, 
        batch_keys,
        expected_state,
        batch_size=batch_size,
        num_history=num_history
      )

      prior_vfe=prior_vfe[0:num_slot].sum(axis=0)
      prior_vfe=prior_vfe[:,-1]
      prior_VFE.append(prior_vfe)

      posterior_vfe=posterior_vfe[0:num_slot].sum(axis=0)
      posterior_vfe=posterior_vfe[:,-1]
      posterior_VFE.append(posterior_vfe)

      bs=bs[0:num_slot].sum(axis=0)#bs.sum(0)
      bs=bs[:,-1]
      kld=jnp.array(kld)
      kld=kld[0:num_slot].sum(axis=0)
      kld=kld[:,-1]
      
      if actions is not None:
        last_action_tmp = jtu.tree_map( lambda x: x[:,-1:], actions)#applied_actions_last = jtu.tree_map( lambda x: x[:,-2:-1], actions)
        last_action=last_action_tmp[-1][-1][-1]
      
      if t<T-1:

        uidx=u_data[t+1+blocknum*4]#u_data[t+1]-1
        oidx=o_data[t+1+blocknum*4]
        
        obs=[0 for _ in range(num_slot+1)]
        obs[-1]=uidx
        obs[uidx]=oidx
        actions_t=jnp.array([uidx for _ in range(num_slot+1)])
        actions_t=jnp.expand_dims(actions_t, -2)
        obs = jtu.tree_map(lambda x: jnp.expand_dims(x, -1).astype(jnp.int32), obs)
        print(f"obs: {obs}")
      
      agents, outcomes, actions, infer_args, batch_keys, neg_efe, pbs,  pbs_pA, pbs_pB,  Hqo, pkld, pfe, oRisk, q_pi,I_B_o,Hqs,pbs_st = update_agent_action_sampling(
            agents, 
            outcomes, 
            actions, 
            infer_args, 
            batch_keys,
            beliefs,
            actions_t,
            reward_fuction=None,
            batch_size=batch_size,
            num_history=num_history
      )

      if t<T-1:
        if uidx==2:
          Q_pi_s.append(np.nan)
        else: 
          Q_pi_s.append(q_pi[0][uidx])

        Q_pi.append(q_pi)
      
      obs_record.append(obs) 

      #情報量記録  
      BS.append(bs)
      KLD.append(kld)
      if t<T-1:
        PBS.append(pbs)
        PKLD.append(pkld)
        PFE.append(pfe)
        Risk.append(oRisk)  
        ambi=pfe-pbs-pkld
        Ambiguity.append(ambi)
        Pv=-oRisk-Hqo
        Pragmaticvalue.append(Pv)
        pBS_pA.append(pbs_pA)
        pBS_pB.append(I_B_o)#
        EFE.append(-1*neg_efe)
        H_qo.append(Hqo)
        Hqs=jnp.array([Hqs[0,0,0],Hqs[0,0,1]])
        H_qs.append(Hqs)
        PBS_st.append(pbs_st)
      selected_slot = int(actions_t[0][num_slot])  
      slot_counts[selected_slot] += 1
      if t<T-1:
        array_selected_slot.append(actions_t[0][num_slot]+1)


    BS = np.array(BS).flatten().tolist()
    KLD = np.array(KLD).flatten().tolist()
    prior_VFE = np.array(prior_VFE).flatten().tolist()
    posterior_VFE = np.array(posterior_VFE).flatten().tolist()


    PBS=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), PBS) 
    PBS=np.array(PBS) 
    PBS=PBS.T
    PBS=PBS.tolist()
    PBS = PBS[:-1]
    PKLD=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), PKLD) 
    PKLD=np.array(PKLD) 
    PKLD=PKLD.T
    PKLD=PKLD.tolist()
    PKLD = PKLD[:-1]
    PFE=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), PFE) 
    PFE=np.array(PFE) 
    PFE=PFE.T
    PFE=PFE.tolist()
    PFE = PFE[:-1]
    Ambiguity=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), Ambiguity) 
    Ambiguity=np.array(Ambiguity) 
    Ambiguity=Ambiguity.T
    Ambiguity=Ambiguity.tolist()
    Ambiguity = Ambiguity[:-1]
    Risk=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), Risk) 
    Risk=np.array(Risk) 
    Risk=Risk.T
    Risk=Risk.tolist()
    Risk = Risk[:-1]
    Pragmaticvalue=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), Pragmaticvalue) 
    Pragmaticvalue=np.array(Pragmaticvalue) 
    Pragmaticvalue=Pragmaticvalue.T
    Pragmaticvalue=Pragmaticvalue.tolist()
    Pragmaticvalue = Pragmaticvalue[:-1]

    EFE=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), EFE) 
    EFE=np.array(EFE)
    EFE=EFE.T
    EFE=EFE.tolist()
    EFE = EFE[:-1]

    pBS_pA=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), pBS_pA) 
    pBS_pA=np.array(pBS_pA) 
    pBS_pA=pBS_pA.T
    pBS_pA=pBS_pA.tolist()
    pBS_pA = pBS_pA[:-1]

    pBS_pB=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), pBS_pB) 
    pBS_pB=np.array(pBS_pB) 
    pBS_pB=pBS_pB.T
    pBS_pB=pBS_pB.tolist()
    pBS_pB = pBS_pB[:-1]

    H_qo=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), H_qo) 
    H_qo=np.array(H_qo) 
    H_qo=H_qo.T
    H_qo=H_qo.tolist()
    H_qo = H_qo[:-1]

    H_qs=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot,)), H_qs)
    H_qs=np.array(H_qs) 
    H_qs=H_qs.T
    H_qs=H_qs.tolist()
    #H_qs = H_qs[:-1]

    PBS_st=jtu.tree_map(lambda x: jnp.reshape(x, (num_slot+1,)), PBS_st) 
    PBS_st=np.array(PBS_st) 
    PBS_st=PBS_st.T
    PBS_st=PBS_st.tolist()
    PBS_st= PBS_st[:-1]

    selected_slot_array=[x.tolist() for x in array_selected_slot]

    PBS_0, PBS_1 = PBS
    PKLD_0, PKLD_1 = PKLD
    PFE_0, PFE_1 = PFE
    Ambiguity_0, Ambiguity_1 = Ambiguity
    Risk_0, Risk_1 = Risk
    Pragmaticvalue_0, Pragmaticvalue_1 = Pragmaticvalue
    EFE_0, EFE_1 = EFE
    pBS_pA_0, pBS_pA_1 = pBS_pA
    pBS_pB_0, pBS_pB_1 = pBS_pB
    H_qo_0, H_qo_1 = H_qo
    PBS_st_0, PBS_st_1 = PBS_st
    H_qs_0, H_qs_1 = H_qs

    PBS_diff = np.abs(np.array(PBS_0) - np.array(PBS_1)).tolist()
    PKLD_diff = np.abs(np.array(PKLD_0) - np.array(PKLD_1)).tolist()
    PFE_diff = np.abs(np.array(PFE_0) - np.array(PFE_1)).tolist()
    Ambiguity_diff = np.abs(np.array(Ambiguity_0) - np.array(Ambiguity_1)).tolist()
    Risk_diff = np.abs(np.array(Risk_0) - np.array(Risk_1)).tolist()
    Pragmaticvalue_diff = np.abs(np.array(Pragmaticvalue_0) - np.array(Pragmaticvalue_1)).tolist()
    EFE_diff = np.abs(np.array(EFE_0) - np.array(EFE_1)).tolist()
    pBS_pA_diff = np.abs(np.array(pBS_pA_0) - np.array(pBS_pA_1)).tolist()
    pBS_pB_diff = np.abs(np.array(pBS_pB_0) - np.array(pBS_pB_1)).tolist()
    H_qo_diff = np.abs(np.array(H_qo_0) - np.array(H_qo_1)).tolist()
    H_qs_diff = np.abs(np.array(H_qs_0) - np.array(H_qs_1)).tolist()
    PBS_st_diff = np.abs(np.array(PBS_st_0) - np.array(PBS_st_1)).tolist()

    PBS_total_diff = np.abs(np.array(PBS_st_0)+np.array(PBS_0) - np.array(PBS_st_1)- np.array(PBS_1)).tolist()

    PBS_select=selected_policy_entropy(np.array(PBS)[:,:30],selected_slot_array)
    Hqo_select=selected_policy_entropy(np.array(H_qo)[:,:30],selected_slot_array)
    I_B_o_select=selected_policy_entropy(np.array(pBS_pB)[:,:30],selected_slot_array)
    PBS_st_select=selected_policy_entropy(np.array(PBS_st)[:,:30],selected_slot_array)
    PBS_total_select=selected_policy_entropy(np.array(PBS_st)+np.array(PBS),selected_slot_array)
    Pragmaticvalue_select=selected_policy_entropy(np.array(Pragmaticvalue)[:,:30],selected_slot_array)
    Hqs_select=selected_policy_entropy(np.array(H_qs)[:,:30],selected_slot_array)

    num_slot_cur = len(PBS)              
    T_cur        = len(PBS[0]) if PBS else 0  

    var_map = {
        "choice":u_data[blocknum*4:blocknum*4+4],
        "reward":o_data[blocknum*4:blocknum*4+4],
        "PBS_diff": PBS_diff,
        "PBS_st_diff":PBS_st_diff,
        "PBS_total_diff":PBS_total_diff,
        "H_qo_diff": H_qo_diff,
        "Pragmaticvalue_diff":Pragmaticvalue_diff,
        "pBS_pA_diff":pBS_pA_diff,
        "pBS_pB_diff":pBS_pB_diff,
        "Q_pi_s":Q_pi_s,
        "PBS_select":PBS_select,
        "PBS_st_select":PBS_st_select,
        "PBS_total_select":PBS_total_select,
        "Hqo_select":Hqo_select,
        "I_B_o_select":I_B_o_select,
        "Pragmaticvalue_select":Pragmaticvalue_select,
        "PBS_0":PBS_0,
        "PBS_1":PBS_1,
        "PBS_st_0":PBS_st_0,
        "PBS_st_1":PBS_st_1,
        "H_qo_0":H_qo_0,
        "H_qo_1":H_qo_1,
        "Pragmaticvalue_0":Pragmaticvalue_0,
        "Pragmaticvalue_1":Pragmaticvalue_1,
        "pBS_pA_0":pBS_pA_0,
        "pBS_pA_1":pBS_pA_1,
        "pBS_pB_0":pBS_pB_0,
        "pBS_pB_1":pBS_pB_1,
        "prior_VFE":prior_VFE,
        "posterior_VFE":posterior_VFE,
        "BS":BS,
        "KLD":KLD,
        "H_qs_diff":H_qs_diff,
        "H_qs_0":H_qs_0,
        "H_qs_1":H_qs_1,
        "Hqs_select":Hqs_select,

    }

    
    for t in range(T_cur):
        row = {
            "p_id": p_id[p],
            "run_num": run_num,
            "blocknum": blocknum,
            "trial": t,     
            "condition":condition,
            "sa1":sa1,
            "sa2":sa2,
            "sb":sb1

        }
        for name, mat in var_map.items():
            row[name] = mat[t]
        records.append(row)

    for name in ["A_jax","B_jax","C_jax", "pB_jax"]:
              if name in locals(): 
                  try: locals()[name] = None
                  except: pass
    jax.clear_caches()

  df_out = pd.DataFrame.from_records(records)
  df_out.to_csv(output_file, index=False)
  print(f"Saved tidy CSV to: {output_file}  (rows={len(df_out)})")


sa1= [12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12]
sa2= [0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 12, 12, 12, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 12, 12, 12, 12, 12, 12, 12, 0, 0, 0,